# Component Classification

CNN over isolated IEEE 315 symbol crops in `component-symbols/`. Structure follows [CircuitNet](https://github.com/aaanthonyyy/CircuitNet)'s classification notebook (`Inspired/circuitnet.md`). The clean val set saturates fast; the number that matters for transfer to real sheets is the *degraded* accuracy at the end.

In [ ]:
import sys
sys.path.insert(0, "..")  # modules live one level up from notebooks/

import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from config import DEFAULT_HP, EPOCHS, OUTPUT_DIR
from data import load_datasets
from model import build_geometric, build_model, compile_model, RandomDegrade
from train import callbacks, lr_schedule, setup
from evaluate import predict, most_confused

In [ ]:
setup()
hp = DEFAULT_HP  # or json.loads((OUTPUT_DIR / "custom_best_hp.json").read_text()) after tune.py
train_ds, val_ds, class_names, class_weights = load_datasets(mixup_alpha=hp["mixup_alpha"])
len(class_names), int(train_ds.cardinality()), int(val_ds.cardinality())

## Augmentation preview

Row 1 raw (already MixUp-blended), row 2 geometric jitter, row 3 scan degradation.

In [ ]:
images, labels = next(iter(train_ds))
geometric, degrade = build_geometric(), RandomDegrade()
rows = [images, geometric(images, training=True), degrade(images, training=True)]

fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for col in range(6):
    for r, batch in enumerate(rows):
        axes[r, col].imshow(batch[col, ..., 0], cmap="gray", vmin=0, vmax=255)
    axes[0, col].set_title(class_names[int(np.argmax(labels[col]))], fontsize=8)
for ax in axes.flat:
    ax.axis("off")

## Model

In [ ]:
model = build_model(len(class_names), hp)  # backbone="efficientnetv2b0" for the pretrained variant
compile_model(model, lr_schedule(int(train_ds.cardinality()), hp["learning_rate"]), hp)
model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks("notebook"),
)

## Curves

In [ ]:
fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(12, 4))
for key in ("accuracy", "val_accuracy", "top3", "val_top3"):
    ax_acc.plot(history.history[key], label=key)
for key in ("loss", "val_loss"):
    ax_loss.plot(history.history[key], label=key)
ax_acc.legend()
ax_loss.legend()

## Validation: clean, test-time augmentation, scan-degraded

In [ ]:
results = {mode: predict(model, val_ds, mode) for mode in ("clean", "tta", "degraded")}
{mode: float(np.mean(y == p)) for mode, (y, p) in results.items()}

In [ ]:
labels, predictions = results["degraded"]  # the harder view is where confusions show
disp = ConfusionMatrixDisplay.from_predictions(labels, predictions, display_labels=class_names, xticks_rotation=90)
disp.figure_.set_size_inches(12, 12)

In [ ]:
print(classification_report(labels, predictions, target_names=class_names, zero_division=0))
most_confused(labels, predictions, class_names)

In [ ]:
model.save("../models/notebook.keras")